# Assignment 2 - Ingénierie des Variables

Nous allons créer de nouvelles variables médicales combinées pour aider nos modèles à mieux apprendre.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/Heart_disease_cleveland_new.csv')
df_feat = df.copy()

## 1. Création de variables cliniques dérivées
- **Risk_Factors_Count** : Somme des anomalies (âge>55, chol>240, etc.)
- **Exercise_Index** : Ratio d'effort cardiaque.

In [ ]:
risk_factors = np.zeros(len(df_feat))
if 'age' in df_feat.columns: risk_factors += (df_feat['age'] > 55).astype(int)
if 'chol' in df_feat.columns: risk_factors += (df_feat['chol'] > 240).astype(int)
if 'trestbps' in df_feat.columns: risk_factors += (df_feat['trestbps'] > 140).astype(int)
if 'fbs' in df_feat.columns: risk_factors += (df_feat['fbs'] == 1).astype(int)
df_feat['Risk_Factors_Count'] = risk_factors

df_feat['Exercise_Index'] = df_feat['thalach'] - (df_feat['oldpeak'] * 10)

def categorize_age(age):
    if age < 40: return 'Young'
    elif age < 60: return 'Middle-Aged'
    else: return 'Senior'
df_feat['Age_Group'] = df_feat['age'].apply(categorize_age)
display(df_feat[['age', 'chol', 'thalach', 'oldpeak', 'Risk_Factors_Count', 'Exercise_Index']].head())

## 2. Encodage et Normalisation (Pipeline)
On transforme les catégories via `get_dummies` et les numériques via `StandardScaler`.

In [ ]:
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal', 'Age_Group']
cont_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'Risk_Factors_Count', 'Exercise_Index']

for c in cat_cols:
    if c in df_feat.columns: df_feat[c] = df_feat[c].astype(str)

target = df_feat['target']
df_feat = df_feat.drop('target', axis=1)

df_trans = pd.get_dummies(df_feat, columns=[c for c in cat_cols if c in df_feat.columns], drop_first=True)
scaler = StandardScaler()
cont_present = [c for c in cont_cols if c in df_trans.columns]
df_trans[cont_present] = scaler.fit_transform(df_trans[cont_present])

df_trans['target'] = target
print(f"Nouvelle dimension : {df_trans.shape}")
display(df_trans.head())